In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "FIFA WC Dataset"
CLEANED_DATA = PROJECT_ROOT / "cleaned_data"
PROCESSED_DATA = PROJECT_ROOT / "processed_data"

team_matches = pd.read_csv(PROCESSED_DATA / "team_matches.csv")
saved_stats = pd.read_csv(PROCESSED_DATA / "team_stats.csv")
results = pd.read_csv(CLEANED_DATA / "results_2000_onwards.csv")


In [2]:
print(results[(results["home_team"] == "United States") &  (results["away_team"] == "Turkey")])

             date      home_team away_team  home_score  away_score  \
9746   29-05-2010  United States    Turkey           2           1   
13725  01-06-2014  United States    Turkey           2           1   
24312  07-06-2025  United States    Turkey           1           2   
25364  25-06-2026  United States    Turkey           2           3   

           tournament           city        country  neutral  
9746         Friendly   Philadelphia  United States    False  
13725        Friendly       Harrison  United States    False  
24312        Friendly  East Hartford  United States    False  
25364  FIFA World Cup      Inglewood  United States    False  


In [3]:
results["date"] = pd.to_datetime(results["date"], dayfirst = True)


In [4]:
results["tournament"].value_counts().head(20)

tournament
Friendly                                8423
FIFA World Cup qualification            5840
UEFA Euro qualification                 1532
African Cup of Nations qualification    1416
UEFA Nations League                      658
AFC Asian Cup qualification              530
African Cup of Nations                   525
FIFA World Cup                           460
CONCACAF Nations League                  422
Gold Cup                                 359
CECAFA Cup                               341
COSAFA Cup                               326
CFU Caribbean Cup qualification          293
Island Games                             283
UEFA Euro                                277
AFC Asian Cup                            256
AFF Championship                         251
Copa América                             248
Gulf Cup                                 189
SAFF Cup                                 135
Name: count, dtype: int64

In [5]:
# result_played will only store the matches which are already played. Future matches are removed (until they happen).
result_played = results[results["home_score"].notna() & results["away_score"].notna()].copy()

In [6]:
INITIAL_ELO = 1500

def get_k_value(tournament):

    if tournament == "Friendly":
        return 10

    elif tournament == "FIFA World Cup":
        return 60

    elif tournament == "FIFA World Cup qualification":
        return 40

    elif tournament in [
        "UEFA Euro",
        "Copa América",
        "AFC Asian Cup",
        "African Cup of Nations",
        "CONCACAF Gold Cup",
        "OFC Nations Cup"
    ]:
        return 35

    elif "qualification" in tournament.lower():
        return 25

    elif "nations league" in tournament.lower():
        return 25

    else:
        return 20
    
elo_ratings = {}

# Created a get_rating function which will get the elo ratings of a team 

def get_rating(team):
    if team not in elo_ratings:
        elo_ratings[team] = INITIAL_ELO
    return elo_ratings[team]

# Created a new get_result function which will store the results Win, Draw, Lose, i.e. W, D, L based on the home-away scores.

def get_result(goals_for, goals_against):
    if goals_for > goals_against:
        return "W"
    elif goals_for < goals_against:
        return "L"
    else:
        return "D"

In [7]:
first_match = result_played.iloc[0]

first_match

date          2000-01-04 00:00:00
home_team                   Egypt
away_team                    Togo
home_score                      2
away_score                      1
tournament               Friendly
city                        Aswan
country                     Egypt
neutral                     False
Name: 0, dtype: object

In [8]:
home_team = first_match["home_team"]
away_team = first_match["away_team"]

home_score = first_match["home_score"]
away_score = first_match["away_score"]

print(home_team, home_score)
print(away_team, away_score)

Egypt 2
Togo 1


In [9]:
home_rating = get_rating(home_team)
away_rating = get_rating(away_team)

print(home_rating)
print(away_rating)

1500
1500


In [10]:
# Calculation if teams draw a match.

expected_home = 1 / (
    1 + 10 ** ((away_rating - home_rating) / 400)
)

expected_away = 1 - expected_home

print(expected_home)
print(expected_away)

0.5
0.5


In [11]:
# Giving points based on match results. if a team Wins it gets 1 point, a Lose gets 0 and 0.5 for a Draw.

if home_score > away_score:
    actual_home = 1
    actual_away = 0

elif home_score < away_score:
    actual_home = 0
    actual_away = 1

else:
    actual_home = 0.5
    actual_away = 0.5

print(actual_home)
print(actual_away)

1
0


In [12]:
# store match-by-match elo history 
history_records = []

In [13]:
# Processing matches in chronological order.

for _, match in result_played.iterrows():

    # Match details
    date = match["date"]
    home_team = match["home_team"]
    away_team = match["away_team"]
    tournament = match["tournament"]
    # Get K value for this tournament
    k = get_k_value(tournament)

    home_score = match["home_score"]
    away_score = match["away_score"]

    # Current ratings
    home_rating = get_rating(home_team)
    away_rating = get_rating(away_team)

    # Expected scores
    expected_home = 1 / (
        1 + 10 ** ((away_rating - home_rating) / 400)
    )
    expected_away = 1 - expected_home

    # Actual scores
    if home_score > away_score:
        actual_home = 1
        actual_away = 0

    elif home_score < away_score:
        actual_home = 0
        actual_away = 1

    else:
        actual_home = 0.5
        actual_away = 0.5

    # New ratings
    new_home_rating = (
        home_rating +
        k * (actual_home - expected_home)
    )

    new_away_rating = (
        away_rating +
        k * (actual_away - expected_away)
    )

    # Results
    home_result = get_result(home_score, away_score)
    away_result = get_result(away_score, home_score)

    # Home team history row
    history_records.append({
        "date": date,
        "team": home_team,
        "opponent": away_team,
        "tournament": tournament,
        "goals_for": home_score,
        "goals_against": away_score,
        "result": home_result,
        "elo_before": home_rating,
        "elo_after": new_home_rating
    })

    # Away team history row
    history_records.append({
        "date": date,
        "team": away_team,
        "opponent": home_team,
        "tournament": tournament,
        "goals_for": away_score,
        "goals_against": home_score,
        "result": away_result,
        "elo_before": away_rating,
        "elo_after": new_away_rating
    })

    # Update ratings dictionary
    elo_ratings[home_team] = new_home_rating
    elo_ratings[away_team] = new_away_rating

In [14]:
len(elo_ratings)

321

In [15]:
elo_ratings_v2 = pd.DataFrame(
    elo_ratings.items(),
    columns=["team", "elo_rating"]
)

elo_ratings_v2 = elo_ratings_v2.sort_values(
    by="elo_rating",
    ascending=False
)
elo_df_v2 = elo_ratings_v2.sort_values(
    by="elo_rating",
    ascending=False
).reset_index(drop=True)

In [16]:
elo_history_v2 = pd.DataFrame(history_records)

In [17]:
elo_ratings_v2["elo_rating"].describe()

count     321.000000
mean     1500.000000
std       171.437520
min       974.319188
25%      1411.248423
50%      1495.331308
75%      1575.251150
max      1992.106077
Name: elo_rating, dtype: float64

In [18]:
# Round history
elo_history_v2["elo_before"] = elo_history_v2["elo_before"].round(2)
elo_history_v2["elo_after"] = elo_history_v2["elo_after"].round(2)

# Round final ratings
elo_ratings_v2["elo_rating"] = elo_ratings_v2["elo_rating"].round(2)

# Add ranking
elo_ratings_v2.insert(
    0,
    "rank",
    range(1, len(elo_ratings_v2) + 1)
)

In [19]:
elo_ratings_v2.head()

,rank,team,elo_rating
101,1,Argentina,1992.11
103,2,France,1985.75
36,3,Spain,1981.76
119,4,Brazil,1918.16
100,5,England,1915.11


In [20]:
elo_ratings_v2.to_csv(PROCESSED_DATA / "elo_ratings_V2.csv", index = False)
elo_history_v2.to_csv(PROCESSED_DATA / "elo_history_v2.csv", index = False)    

In [21]:
print(elo_history_v2.shape)
print(elo_ratings_v2.shape)

(50772, 9)
(321, 3)


## Elo V2 Complete

Built a tournament-weighted Elo system by replacing the fixed K-factor with competition-specific K values. Higher importance tournaments (World Cup, qualifiers, continental championships) now have greater influence on rating changes than friendlies.

Outputs:
- `elo_ratings_v2.csv`
- `elo_history_v2.csv`

Next Step: Evaluate V1 vs V2 rankings and explore goal-difference weighting (V2.5).

The biggest difference i like to point out is that the Version 1 of the elo ratings and elo history were based purely on the K Value which is 20. elo ratings_v2 and elo_history_v2 is based on Tournament values. if its a friendly or a very lower level tournament, then the team will get less elo points compared to the one participating in World Cup and continental cups and qualifications.